# Supply Chain Performance Analytics — EDA Notebook
This notebook uses the supplied `data/supply_chain_data.csv`. It follows the requirement document tools: Pandas, NumPy, SciPy, statsmodels, Matplotlib and Seaborn.

**Limitations:** no order/delivery dates or explicit on-time status are present, so actual on-time delivery rate, order cycle time and time-series demand forecasting cannot be calculated from this file.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf
sns.set_theme(style='whitegrid')
Path('outputs/figures').mkdir(parents=True, exist_ok=True)

## 1. Load and inspect data

In [ ]:
df = pd.read_csv('data/supply_chain_data.csv')
df.head()

In [ ]:
print('Shape:', df.shape)
display(df.info())
display(df.isna().sum().sort_values(ascending=False))
display(df.describe(include='all').T)

## 2. Clean data

In [ ]:
df.columns = df.columns.astype(str).str.strip()
for c in df.select_dtypes(include=['object','string']).columns:
    df[c] = df[c].astype('string').str.strip()
numeric_cols = ['Price','Availability','Number of products sold','Revenue generated','Stock levels','Lead times','Order quantities','Shipping times','Shipping costs','Lead time','Production volumes','Manufacturing lead time','Manufacturing costs','Defect rates','Costs']
for c in numeric_cols:
    if c in df.columns: df[c] = pd.to_numeric(df[c], errors='coerce')
df['Low stock flag'] = df['Stock levels'].le(10)
df['Sales per available unit'] = np.where(df['Availability'] > 0, df['Number of products sold']/df['Availability'], np.nan)
df.to_csv('outputs/cleaned_supply_chain_data.csv', index=False)
df.head()

## 3. KPI summary

In [ ]:
kpis = pd.DataFrame({
 'KPI':['SKU count','Total units sold','Total revenue','Average supplier lead time','Supplier lead-time standard deviation','Average shipping time','Average shipping cost','Average manufacturing lead time','Average defect-rate field','Low-stock rows (<=10)'],
 'Value':[df['SKU'].nunique(),df['Number of products sold'].sum(),df['Revenue generated'].sum(),df['Lead time'].mean(),df['Lead time'].std(),df['Shipping times'].mean(),df['Shipping costs'].mean(),df['Manufacturing lead time'].mean(),df['Defect rates'].mean(),df['Stock levels'].le(10).sum()]})
display(kpis)
kpis.to_csv('outputs/kpi_summary.csv', index=False)

## 4. Exploratory visualizations

In [ ]:
plt.figure(figsize=(9,5)); sns.barplot(data=df, x='Product type', y='Revenue generated', ci=None, color='C0'); plt.xticks(rotation=25); plt.title('Revenue by Product Type'); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(9,5)); sns.barplot(data=df, x='Supplier name', y='Lead time', ci=None, color='C0'); plt.xticks(rotation=35, ha='right'); plt.title('Lead Time by Supplier'); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(9,5)); sns.barplot(data=df, x='Transportation modes', y='Shipping costs', ci=None, color='C0'); plt.title('Shipping Costs by Transportation Mode'); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(8,5)); sns.scatterplot(data=df, x='Availability', y='Number of products sold', hue='Product type'); plt.title('Sales vs Availability'); plt.tight_layout(); plt.show()

## 5. Statistical analysis (SciPy)

In [ ]:
supplier_groups = [g['Lead time'].dropna().values for _,g in df.groupby('Supplier name') if g['Lead time'].notna().sum() >= 2]
if len(supplier_groups) >= 2:
    f_stat, p_value = stats.f_oneway(*supplier_groups)
    print('ANOVA lead time by supplier: F=', f_stat, 'p=', p_value)
else:
    print('Not enough data for ANOVA')
pair = df[['Shipping times','Shipping costs']].dropna()
if len(pair)>=3 and pair.nunique().min()>1:
    print('Shipping time vs cost Pearson:', stats.pearsonr(pair['Shipping times'], pair['Shipping costs']))

## 6. Regression example (statsmodels)

In [ ]:
reg = df[['Shipping costs','Shipping times','Order quantities','Price','Lead time']].dropna()
if len(reg) >= 10:
    model = smf.ols('Q("Shipping costs") ~ Q("Shipping times") + Q("Order quantities") + Q("Price") + Q("Lead time")', data=reg).fit()
    print(model.summary())
else:
    print('Not enough complete rows for regression')

## 7. Export findings
Use `outputs/kpi_summary.csv` and the cleaned CSV for Power BI. Do not claim on-time delivery or order-cycle KPIs because the required dates/status fields are absent. The dataset also lacks a date-indexed sales history, so ARIMA or time-series forecasting is not justified without additional dated records.